In [7]:
# Imports
# Use mostly numpy for script performance
import numpy as np
import pandas as pd

# 1. Load data
data = pd.read_csv('trackingData_v2.csv')

# Convert numeric columns to proper types
data['Location.X'] = pd.to_numeric(data['Location.X'], errors='coerce')
data['Location.Y'] = pd.to_numeric(data['Location.Y'], errors='coerce')
data['UTCTime'] = pd.to_numeric(data['UTCTime'], errors='coerce')

# 2. Filter out the necessary data
valid_data = data[data['ClockState'] == 'ClockStateRunning'].copy()

# 3. Calculate differences for consecutive rows for locations and time
valid_data['dx'] = valid_data.groupby('player')['Location.X'].diff()
valid_data['dy'] = valid_data.groupby('player')['Location.Y'].diff()
valid_data['dt'] = valid_data.groupby('player')['UTCTime'].diff()

# 4. Calculate distance, speed, and acceleration
valid_data['distance'] = np.sqrt(valid_data['dx']**2 + valid_data['dy']**2)
valid_data['speed'] = valid_data['distance'] / valid_data['dt']
valid_data['acceleration'] = valid_data.groupby('player')['speed'].diff() / valid_data['dt']

# 5. Convert OnPlayingSurface from TRUE/FALSE to 1/0
valid_data['OnPlayingSurface'] = valid_data['OnPlayingSurface'].astype(int)

# 6. Identifying shifts using diff
valid_data['shift_change'] = valid_data['OnPlayingSurface'].diff().fillna(0)
valid_data['start_of_shift'] = (valid_data['shift_change'] == 1).astype(int)
valid_data['shift_id'] = valid_data['start_of_shift'].cumsum()

# 7. Determine Statistics for Each Player
players_stats = {}
for player in valid_data['player'].unique():
    player_data = valid_data[valid_data['player'] == player]
    
    # Total distance
    total_distance = player_data['distance'].sum()
    
    # Average speed per shift
    average_speed_per_shift = player_data.groupby('shift_id')['speed'].mean()
    
    # Max speed and acceleration
    max_speed = player_data['speed'].max()
    max_acceleration = player_data['acceleration'].max()
    
    # Print player stats
    print(f"\nPlayer {player} stats:")
    print(f"Total Distance: {total_distance:.4f} feet")
    print(f"Max Speed: {max_speed:.4f} ft/s")
    print(f"Max Acceleration: {max_acceleration:.4f} ft/s^2")
    print(f"Average Speed per shift (ft/s):")
    for idx, avg_speed in enumerate(average_speed_per_shift.values, 1):
        print(f"Shift {idx}: {avg_speed:.4f}")
    
    players_stats[player] = {
        'avg_speed_per_shift': average_speed_per_shift.values,
        'max_speed': max_speed,
        'max_acceleration': max_acceleration,
        'total_distance': total_distance
    }

# 8. Rank players based on acceleration first and then speed
ranking = sorted(players_stats.keys(), 
                 key=lambda x: (players_stats[x]['max_acceleration'], players_stats[x]['max_speed']), reverse=True)

print(f"\nRanking from best to worst skater: {ranking}")


Player nan stats:
Total Distance: 0.0000 feet
Max Speed: nan ft/s
Max Acceleration: nan ft/s^2
Average Speed per shift (ft/s):

Player Player1 stats:
Total Distance: 1.9314 feet
Max Speed: 0.0163 ft/s
Max Acceleration: -0.0000 ft/s^2
Average Speed per shift (ft/s):
Shift 1: 0.0150

Player Player2 stats:
Total Distance: 1.0797 feet
Max Speed: 0.0096 ft/s
Max Acceleration: -0.0000 ft/s^2
Average Speed per shift (ft/s):
Shift 1: 0.0086

Player Player3 stats:
Total Distance: 1.1376 feet
Max Speed: 0.0093 ft/s
Max Acceleration: 0.0000 ft/s^2
Average Speed per shift (ft/s):
Shift 1: 0.0087

Player Player4 stats:
Total Distance: 1.3904 feet
Max Speed: 0.0122 ft/s
Max Acceleration: 0.0001 ft/s^2
Average Speed per shift (ft/s):
Shift 1: 0.0099

Player Player5 stats:
Total Distance: 1.3467 feet
Max Speed: 0.0125 ft/s
Max Acceleration: 0.0002 ft/s^2
Average Speed per shift (ft/s):
Shift 1: 0.0095

Player Player6 stats:
Total Distance: 1.2276 feet
Max Speed: 0.0109 ft/s
Max Acceleration: 0.0001 f